# T1.5 - Retention-branch zero-out ablation

This notebook loads each pre-trained `RetNet` / `DualBranch` / `SpatialMult` / `SpatialAdd` / `SpatialMultRetNet` aggregator checkpoint, runs inference on the held-out fold's test set with `M_retention = 0` forced in the forward pass (CLAM softmax-weighted-sum branch only), and computes the QWK against the original RetNet checkpoint's full-branch QWK.

**Output:** `retention_ablation_per_fold.csv` (per-fold, per-variant full vs zero-retention QWK + DeltaQWK) and `retention_ablation_summary.csv` (cohort aggregation: mean QWK across 5 folds, paired-t p-value, Cohen's d).

**Purpose:** Closes thesis review T1.5. Empirically tests whether the retention branch contributes to the QWK gain OR whether the CLAM softmax-weighted-sum branch alone (with the same TopK=128 patches) carries all the performance and the retention branch is effectively inert due to the rank-1 attention dominance shown in Section 3.6 (rank-1 >= 0.9969 on the three example slides; cohort-wide via the T2.9 sweep).

**Variants covered (the 5 retention-enabled aggregators):**
- `DualBranch` - retention branch is `h_topk.mean(dim=0)` (no learned module; tests TopK-mean contribution)
- `RetNet` - retention branch is non-spatial scaled dot-product retention with full component stack
- `SpatialMult` - standalone spatial retention with multiplicative gamma decay
- `SpatialAdd` - standalone spatial retention with additive gamma decay
- `SpatialMultRetNet` - spatial retention + full RetentionBlock stack

`ABMIL` is single-branch (no retention) and is excluded from the ablation but loaded as a control to confirm the loader works.

**Runtime:** approximately 4-8 hours on a Colab T4 GPU. 5 variants x 2 hp x 5 folds x 2 encoders = 100 zero-retention sweeps (~1,803 slides each). Full-branch QWK is read directly from `predictions/*.json` so only the zero-retention forward needs new compute. Use `TEST_MODE = True` for a sanity check on `RetNet_bestHP / UNI / fold 0` (~2-3 min).

**Resume:** the notebook writes per-`(encoder, variant, hp, fold)` partial CSVs to `PARTIAL_DIR`. If Colab disconnects, re-run from Cell 1 onward; completed chunks are skipped automatically.

**Surgical change:** the forward path is duplicated as `forward_zero_retention(model, features, coords)` which reproduces `DualBranchModel.forward` up to the `M_clam + M_retention` fusion, then zeroes out `M_retention` before classification. The original `model.state_dict()` is NEVER modified - only the forward call is replaced for the ablation pass.

## Setup: paths + Drive mount

**Mohamed should edit `DRIVE_ROOT` to match his Drive layout.** The defaults match the layout used by `extract_rank1_attention_colab.ipynb` (T2.9 sweep). Outputs go to `MyDrive/CLAM_TR_Results/v20_fullscale/retention_ablation/`.

In [ ]:
# T1.5 step 1: mount Drive, set paths.
from google.colab import drive
drive.mount('/content/drive')

import os

# EDIT THIS to match your Drive layout.
DRIVE_ROOT = '/content/drive/MyDrive'

# Feature directories (one per encoder).
FEATURE_DIRS = {
    'UNI':    os.path.join(DRIVE_ROOT, 'UNI_features'),         # .pt files, 1024d
    'UNI2-h': os.path.join(DRIVE_ROOT, 'UNI_PANDA_Features'),   # .h5 files, 1536d
}
FILE_EXTS  = {'UNI': '.pt', 'UNI2-h': '.h5'}
INPUT_DIMS = {'UNI': 1024, 'UNI2-h': 1536}

# CLAM-TR v20 results root (contains UNI/ and UNI2-h/ subdirs).
RESULTS_ROOT = os.path.join(DRIVE_ROOT, 'CLAM_TR_Results', 'v20_fullscale')

# T1.5 outputs go here.
T15_OUTPUT_DIR = os.path.join(RESULTS_ROOT, 'retention_ablation')
PARTIAL_DIR    = os.path.join(T15_OUTPUT_DIR, 'partial_per_fold')
os.makedirs(PARTIAL_DIR, exist_ok=True)

# Sanity check: confirm PARTIAL_DIR exists and is writeable.
# If makedirs silently failed (e.g. Drive mount issue or read-only path),
# this fails LOUD instead of letting the main loop crash later.
assert os.path.isdir(PARTIAL_DIR), (
    f'PARTIAL_DIR does not exist after makedirs: {PARTIAL_DIR}. '
    f'Check that DRIVE_ROOT is correct and writeable. '
    f'Try: ls -la {os.path.dirname(T15_OUTPUT_DIR)}')
_probe = os.path.join(PARTIAL_DIR, '.write_probe')
try:
    with open(_probe, 'w') as _f:
        _f.write('probe')
    os.remove(_probe)
except OSError as e:
    raise OSError(f'PARTIAL_DIR is not writeable: {PARTIAL_DIR}. Drive mount issue? {e}')


# Per-fold CSV (concatenation of all partial files).
PER_FOLD_CSV = os.path.join(T15_OUTPUT_DIR, 'retention_ablation_per_fold.csv')
# Aggregate summary CSV.
SUMMARY_CSV  = os.path.join(T15_OUTPUT_DIR, 'retention_ablation_summary.csv')

# =====================================================
# TEST MODE - run only fold 0 of RetNet_bestHP on UNI as
# a sanity check. Flip to False for the full sweep.
# =====================================================
TEST_MODE = True

print(f'DRIVE_ROOT   = {DRIVE_ROOT}')
print(f'RESULTS_ROOT = {RESULTS_ROOT}')
print(f'T15_OUTPUT   = {T15_OUTPUT_DIR}')
print(f'TEST_MODE    = {TEST_MODE}')

In [ ]:
# T1.5 step 2: install Python dependencies.
# (torch + numpy + pandas + scikit-learn are preinstalled on Colab; add h5py + tqdm.)
import subprocess, sys
for pkg in ['h5py', 'tqdm', 'scikit-learn', 'scipy']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

import torch, numpy as np, pandas as pd, h5py, json, time
from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score
from scipy import stats as scipy_stats
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}, device={device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}, '
          f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Reproducibility (matches v20 training).
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

## Model class definitions (extracted from v20 notebook cells 21+23+25)

All classes below are extracted verbatim from `CLAM-master/final-code/04-final-v20/CLAM_TR_v20_FullScale_Colab.ipynb`. Same source as `extract_rank1_attention_colab.ipynb` (T2.9). No structural changes; only the training-loop hooks are removed.

Critical for T1.5: the `DualBranchModel.forward` (cell 6 below) is the canonical full-branch path; the ablation forward in cell 8 replicates this path up to the `M_clam + M_retention` fusion and zeroes out `M_retention` before classification.

In [ ]:
# T1.5 step 3: model classes extracted verbatim from
# CLAM_TR_v20_FullScale_Colab.ipynb cells 21 (components) + 23 (classes) + 25 (factory).
import torch
import torch.nn as nn
import torch.nn.functional as F


class GatedAttention(nn.Module):
    """v20 exact. Gated attention with temperature scaling."""
    def __init__(self, input_dim=512, hidden_dim=256, dropout=0.25):
        super().__init__()
        self.attention_a = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Tanh(), nn.Dropout(dropout))
        self.attention_b = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Sigmoid(), nn.Dropout(dropout))
        self.attention_c = nn.Linear(hidden_dim, 1)

    def forward(self, x, temperature=1.0):
        a = self.attention_a(x)
        b = self.attention_b(x)
        return self.attention_c(a * b).squeeze(-1) / temperature


class SwishGate(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        return x * torch.sigmoid(self.proj(x))


class FFNBlock(nn.Module):
    def __init__(self, dim, expansion=4, dropout=0.1):
        super().__init__()
        hidden = dim * expansion
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout))
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return x + self.net(self.norm(x))


class RetentionHead(nn.Module):
    """Scaled dot-product attention (no spatial decay)."""
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        self.head_dim = hidden_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_K = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, self.head_dim, bias=False)

    def forward(self, x):
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class SpatialRetentionHeadMult(nn.Module):
    """Multiplicative spatial decay: scores * D."""
    def __init__(self, hidden_dim, gamma, num_heads):
        super().__init__()
        self.head_dim = hidden_dim // num_heads
        self.gamma = gamma
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_K = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, self.head_dim, bias=False)

    def forward(self, x, coords):
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        diff = coords.unsqueeze(0) - coords.unsqueeze(1)
        distances = torch.norm(diff.float(), dim=-1)
        normalized = distances / (distances.max() + 1e-6)
        D = self.gamma ** normalized
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale * D
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class SpatialRetentionHeadAdd(nn.Module):
    """Additive spatial decay: scores + log(D)."""
    def __init__(self, hidden_dim, gamma, num_heads):
        super().__init__()
        self.head_dim = hidden_dim // num_heads
        self.gamma = gamma
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_K = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, self.head_dim, bias=False)

    def forward(self, x, coords):
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        diff = coords.unsqueeze(0) - coords.unsqueeze(1)
        distances = torch.norm(diff.float(), dim=-1)
        normalized = distances / (distances.max() + 1e-6)
        D = self.gamma ** normalized
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        scores = scores + torch.log(D + 1e-6)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class SpatialRetentionMult(nn.Module):
    """Standalone spatial retention (multiplicative). NO dropout/norm/residual."""
    def __init__(self, hidden_dim=512, num_heads=8, gamma_values=None):
        super().__init__()
        assert gamma_values is not None and len(gamma_values) == num_heads
        self.heads = nn.ModuleList([
            SpatialRetentionHeadMult(hidden_dim, gamma_values[i], num_heads)
            for i in range(num_heads)])
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, coords):
        head_outputs = [h(x, coords) for h in self.heads]
        return self.output_proj(torch.cat(head_outputs, dim=-1))


class SpatialRetentionAdd(nn.Module):
    def __init__(self, hidden_dim=512, num_heads=8, gamma_values=None):
        super().__init__()
        assert gamma_values is not None and len(gamma_values) == num_heads
        self.heads = nn.ModuleList([
            SpatialRetentionHeadAdd(hidden_dim, gamma_values[i], num_heads)
            for i in range(num_heads)])
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, coords):
        head_outputs = [h(x, coords) for h in self.heads]
        return self.output_proj(torch.cat(head_outputs, dim=-1))


class RetentionBlock(nn.Module):
    """Enhanced retention with full component stack.
    Order: MHA -> output_proj -> dropout -> Norm -> Residual -> Gate -> FFN"""
    def __init__(self, hidden_dim=512, num_heads=8, gamma_values=None,
                 use_spatial=False, spatial_mode='multiplicative',
                 use_norm=True, norm_type='group', num_groups=8,
                 use_residual=True, use_gate=True, use_ffn=True,
                 ffn_expansion=4, dropout=0.25):
        super().__init__()
        self.use_spatial = use_spatial
        self.use_norm = use_norm
        self.use_residual = use_residual
        self.use_gate = use_gate
        self.use_ffn = use_ffn
        if use_spatial:
            assert gamma_values is not None and len(gamma_values) == num_heads
            HeadClass = (SpatialRetentionHeadMult if spatial_mode == 'multiplicative'
                         else SpatialRetentionHeadAdd)
            self.heads = nn.ModuleList([
                HeadClass(hidden_dim, gamma_values[i], num_heads)
                for i in range(num_heads)])
        else:
            self.heads = nn.ModuleList([
                RetentionHead(hidden_dim, num_heads) for _ in range(num_heads)])
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        if use_norm:
            self.norm = (nn.GroupNorm(num_groups, hidden_dim)
                         if norm_type == 'group' else nn.LayerNorm(hidden_dim))
        if use_gate:
            self.gate = SwishGate(hidden_dim)
        if use_ffn:
            self.ffn = FFNBlock(hidden_dim, ffn_expansion, dropout)

    def forward(self, x, coords=None):
        identity = x
        if self.use_spatial:
            head_outputs = [h(x, coords) for h in self.heads]
        else:
            head_outputs = [h(x) for h in self.heads]
        out = self.output_proj(torch.cat(head_outputs, dim=-1))
        out = self.dropout(out)
        if self.use_norm:     out = self.norm(out)
        if self.use_residual: out = out + identity
        if self.use_gate:     out = self.gate(out)
        if self.use_ffn:      out = self.ffn(out)
        return out


class ABMIL(nn.Module):
    """Single-branch baseline. Uses ALL patches. No retention branch (excluded from T1.5)."""
    def __init__(self, input_dim, hidden_dim=512, n_classes=6, dropout=0.25):
        super().__init__()
        self.needs_coords = False
        self.feature_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.attention = GatedAttention(hidden_dim, hidden_dim // 2, dropout)
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, coords=None, temperature=1.0):
        h = self.feature_transform(x)
        A = F.softmax(self.attention(h, temperature), dim=0)
        M = torch.sum(A.unsqueeze(-1) * h, dim=0)
        return self.classifier(M)


class DualBranchModel(nn.Module):
    """Unified dual-branch model. Covers all non-ABMIL experiments.

    Forward path (full-branch, training-time):
        h          = feature_transform(x)                 [N, hidden_dim]
        A          = attention(h, T)                      [N]
        A_softmax  = F.softmax(A, dim=0)                  [N]
        topk_idx   = torch.topk(A, K).indices             [K]
        h_topk     = h[topk_idx]                          [K, hidden_dim]
        A_topk     = A_softmax[topk_idx]                  [K]
        M_clam     = sum(A_topk.unsqueeze(-1) * h_topk)   [hidden_dim]
        if retention_module is None:
            M_retention = h_topk.mean(dim=0)              [hidden_dim] (DualBranch only)
        else:
            M_retention = retention_module(h_topk[,coords_topk]).mean(dim=0)
        logits     = classifier(M_clam + M_retention)     [n_classes]

    For T1.5, we replicate this path in `forward_zero_retention` and replace
    `M_retention` with `torch.zeros_like(M_clam)` BEFORE the classifier call.
    """
    def __init__(self, input_dim, hidden_dim=512, n_classes=6, dropout=0.25,
                 K=128, retention_module=None, needs_coords=False):
        super().__init__()
        self.K = K
        self.needs_coords = needs_coords
        self.feature_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.attention = GatedAttention(hidden_dim, hidden_dim // 2, dropout)
        self.retention_module = retention_module
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, coords=None, temperature=1.0):
        h = self.feature_transform(x)
        A = self.attention(h, temperature)
        A_softmax = F.softmax(A, dim=0)
        K_actual = min(self.K, h.size(0))
        topk_indices = torch.topk(A, K_actual).indices
        h_topk = h[topk_indices]
        A_topk = A_softmax[topk_indices]
        M_clam = torch.sum(A_topk.unsqueeze(-1) * h_topk, dim=0)
        if self.retention_module is None:
            M_retention = h_topk.mean(dim=0)
        elif self.needs_coords:
            coords_topk = coords[topk_indices]
            M_retention = self.retention_module(h_topk, coords_topk).mean(dim=0)
        else:
            M_retention = self.retention_module(h_topk).mean(dim=0)
        return self.classifier(M_clam + M_retention)


# Hyperparameter sets (verbatim from v20 cell 6).
HP_SETS = {
    'default': {
        'dropout': 0.25,
        'gamma_values': [0.80, 0.85, 0.90, 0.93, 0.95, 0.97, 0.98, 0.99],
    },
    'best': {
        'dropout': 0.15,
        'gamma_values': [0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 0.99, 0.995],
    },
}

# Architecture constants (v20 CONFIG).
HIDDEN_DIM    = 512
N_CLASSES     = 6
NUM_HEADS     = 8
K_TOPK        = 128
FFN_EXPANSION = 4
NUM_GROUPS    = 8


def create_model(experiment_name, hp_name, input_dim):
    """Factory matching v20 cell 25. Returns nn.Module ready for state_dict load."""
    hp = HP_SETS[hp_name]
    dropout = hp['dropout']
    gamma_values = hp['gamma_values']
    if experiment_name.startswith('ABMIL'):
        return ABMIL(input_dim, HIDDEN_DIM, N_CLASSES, dropout)
    if experiment_name.startswith('DualBranch'):
        return DualBranchModel(input_dim, HIDDEN_DIM, N_CLASSES, dropout,
                               K_TOPK, retention_module=None, needs_coords=False)
    if experiment_name.startswith('RetNet'):
        retention = RetentionBlock(
            hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS, use_spatial=False,
            use_norm=True, norm_type='group', num_groups=NUM_GROUPS,
            use_residual=True, use_gate=True, use_ffn=True,
            ffn_expansion=FFN_EXPANSION, dropout=dropout)
        return DualBranchModel(input_dim, HIDDEN_DIM, N_CLASSES, dropout,
                               K_TOPK, retention_module=retention, needs_coords=False)
    if experiment_name.startswith('SpatialMultRetNet'):
        retention = RetentionBlock(
            hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS, gamma_values=gamma_values,
            use_spatial=True, spatial_mode='multiplicative',
            use_norm=True, norm_type='group', num_groups=NUM_GROUPS,
            use_residual=True, use_gate=True, use_ffn=True,
            ffn_expansion=FFN_EXPANSION, dropout=dropout)
        return DualBranchModel(input_dim, HIDDEN_DIM, N_CLASSES, dropout,
                               K_TOPK, retention_module=retention, needs_coords=True)
    if experiment_name.startswith('SpatialMult_') or experiment_name.startswith('SpatialMult/') or experiment_name == 'SpatialMult' or (experiment_name.startswith('SpatialMult') and 'RetNet' not in experiment_name):
        spatial = SpatialRetentionMult(
            hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS, gamma_values=gamma_values)
        return DualBranchModel(input_dim, HIDDEN_DIM, N_CLASSES, dropout,
                               K_TOPK, retention_module=spatial, needs_coords=True)
    if experiment_name.startswith('SpatialAdd'):
        spatial = SpatialRetentionAdd(
            hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS, gamma_values=gamma_values)
        return DualBranchModel(input_dim, HIDDEN_DIM, N_CLASSES, dropout,
                               K_TOPK, retention_module=spatial, needs_coords=True)
    raise ValueError(f'Unknown experiment: {experiment_name}')

print('Model classes + factory defined.')

## Ablation forward pass

**The surgical change.** `forward_zero_retention(model, features, coords)` replicates the `DualBranchModel.forward` path up to the `M_clam + M_retention` fusion, then **replaces `M_retention` with `torch.zeros_like(M_clam)` before the classifier call**.

**Why a separate function instead of a forward hook or monkey-patch?**
1. **Safer**: no risk of accidentally mutating `model.classifier`'s parameters or state_dict. The model weights stay untouched.
2. **Cleaner**: the addition operation `M_clam + M_retention` happens inline in `DualBranchModel.forward` (line `return self.classifier(M_clam + M_retention)`); a forward hook on `model.classifier` would see the post-addition tensor, requiring us to reverse-engineer `M_clam` from it, which is impossible.
3. **Reproducible**: every dependent computation (feature_transform, attention, softmax, topk, retention_module) is identical to `DualBranchModel.forward` byte-for-byte; only the fusion line differs.

**ABMIL fallback.** For `ABMIL` (single-branch), `forward_zero_retention` returns the standard forward output - there is no retention branch to zero. ABMIL is included only as a control to confirm the loader/inference pipeline works; its zero-retention QWK equals its full-branch QWK by construction.

**Determinism.** `torch.no_grad()` and `model.eval()` (which disables dropout and freezes BN/GN running stats) are applied. Temperature is fixed at `1.0` to match the v20 inference path.

In [ ]:
# T1.5 step 4: ablation forward pass.
import torch
import torch.nn.functional as F


@torch.no_grad()
def forward_zero_retention(model, features, coords, temperature=1.0):
    """Replicate DualBranchModel.forward but ZERO OUT M_retention before classification.

    For ABMIL (single-branch) the standard forward path is used - there is no
    retention branch to zero. Returns int (argmax class).

    Args:
        model:       eval()-mode nn.Module (ABMIL or DualBranchModel) - NOT MUTATED
        features:    float tensor [N, input_dim] on device
        coords:      float tensor [N, 2] on device (used only by spatial variants)
        temperature: gated-attention softmax temperature (1.0 matches v20 inference)

    Returns:
        pred_zero: int, argmax of logits with M_retention set to zero
    """
    # ABMIL has no retention branch - fall through to its standard forward.
    if isinstance(model, ABMIL):
        logits = model(features, coords, temperature)
        return int(torch.argmax(logits, dim=-1).item())

    # DualBranchModel surgical path.
    assert isinstance(model, DualBranchModel), f'Unexpected model type: {type(model)}'
    h = model.feature_transform(features)
    A = model.attention(h, temperature)
    A_softmax = F.softmax(A, dim=0)
    K_actual = min(model.K, h.size(0))
    topk_indices = torch.topk(A, K_actual).indices
    h_topk = h[topk_indices]
    A_topk = A_softmax[topk_indices]
    M_clam = torch.sum(A_topk.unsqueeze(-1) * h_topk, dim=0)
    # --- SURGICAL EDIT: replace M_retention with zeros ---
    M_retention = torch.zeros_like(M_clam)
    # ------------------------------------------------------
    logits = model.classifier(M_clam + M_retention)  # equivalent to model.classifier(M_clam)
    return int(torch.argmax(logits, dim=-1).item())


@torch.no_grad()
def forward_full(model, features, coords, temperature=1.0):
    """Standard full-branch forward pass (used as a verification fallback if the
    cached predictions JSON is missing for a fold). Returns int (argmax class)."""
    logits = model(features, coords, temperature)
    return int(torch.argmax(logits, dim=-1).item())

print('Ablation forward pass defined.')

In [ ]:
# T1.5 step 5: data loader (matches v20 cell 19 _load_one + T2.9 cell 7).
import os, json, h5py
import numpy as np
import torch


def load_slide_features(slide_id, features_dir, file_ext):
    """Load one slide's (features, coords) tensors. Matches v20 PANDADataset._load_one.

    Returns (features, coords) or (None, None) if the file is missing/malformed.
    """
    fpath = os.path.join(features_dir, f'{slide_id}{file_ext}')
    if not os.path.exists(fpath):
        return None, None
    try:
        if file_ext == '.pt':
            data = torch.load(fpath, map_location='cpu', weights_only=False)
            if isinstance(data, dict):
                features = data.get('features',
                                    data.get('embeddings', list(data.values())[0]))
                features = (torch.tensor(features, dtype=torch.float32)
                            if isinstance(features, np.ndarray) else features.float())
                if features.ndim == 3:
                    features = features.squeeze(0)
                coords_raw = data.get('coords', data.get('coordinates', None))
                if coords_raw is None:
                    coords = torch.zeros(features.shape[0], 2, dtype=torch.float32)
                else:
                    coords = (torch.tensor(coords_raw, dtype=torch.float32)
                              if isinstance(coords_raw, np.ndarray)
                              else coords_raw.float())
                    if coords.ndim == 3:
                        coords = coords.squeeze(0)
            elif isinstance(data, torch.Tensor):
                features = data.float()
                if features.ndim == 3:
                    features = features.squeeze(0)
                coords = torch.zeros(features.shape[0], 2, dtype=torch.float32)
        elif file_ext == '.h5':
            with h5py.File(fpath, 'r') as f:
                features = torch.tensor(f['features'][:], dtype=torch.float32).squeeze(0)
                coords   = torch.tensor(f['coords'][:],   dtype=torch.float32).squeeze(0)
        else:
            raise ValueError(f'Unknown extension: {file_ext}')
    except Exception as e:
        print(f'  WARN: failed to load {slide_id}: {e}')
        return None, None
    return features, coords


def load_predictions_file(encoder, variant, hp, fold):
    """Return (slide_ids, full_predictions, labels, path) for the given fold.

    Reads {RESULTS_ROOT}/{encoder}/predictions/{variant}_{hp}HP_fold{N}_predictions.json.
    The full-branch predictions are cached here from the v20 training run, so we can
    compute the full QWK without re-running the full forward pass.
    """
    pred_path = os.path.join(
        RESULTS_ROOT, encoder, 'predictions',
        f'{variant}_{hp}HP_fold{fold}_predictions.json')
    if not os.path.exists(pred_path):
        return None, None, None, pred_path
    with open(pred_path) as f:
        pred = json.load(f)
    return (pred['slide_ids'],
            np.asarray(pred['predictions'], dtype=int),
            np.asarray(pred['labels'], dtype=int),
            pred_path)


def load_checkpoint(ckpt_path, model, device):
    """Load a v20 checkpoint (.pt) into `model`. Tolerates several state_dict layouts."""
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(state, dict) and 'model_state_dict' in state:
        sd = state['model_state_dict']
    elif isinstance(state, dict) and 'state_dict' in state:
        sd = state['state_dict']
    elif isinstance(state, dict) and all(isinstance(v, torch.Tensor) for v in state.values()):
        sd = state
    else:
        raise ValueError(f'Unrecognised checkpoint format at {ckpt_path}: '
                         f'top-level keys = {list(state.keys()) if isinstance(state, dict) else type(state)}')
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:
        print(f'  WARN: missing keys in checkpoint: {missing[:5]}...')
    if unexpected:
        print(f'  WARN: unexpected keys in checkpoint: {unexpected[:5]}...')
    return model

print('Data loader + checkpoint loader defined.')

## Main ablation loop

Loops over `(encoder, variant, hp, fold)` for the 5 retention-enabled variants x 2 hp x 5 folds x 2 encoders = **100 zero-retention sweeps**. For each:

1. Resolve checkpoint at `{RESULTS_ROOT}/{encoder}/checkpoints/{variant}_{hp}HP_fold{N}_epoch.pt`.
2. Read cached full-branch predictions + labels from `{RESULTS_ROOT}/{encoder}/predictions/{variant}_{hp}HP_fold{N}_predictions.json`. Use these to compute `full_qwk = cohen_kappa_score(labels, full_preds, weights='quadratic')` without re-running compute.
3. Instantiate model, load weights, set to `eval()` mode.
4. For each test slide: load features, run `forward_zero_retention(...)`, record `zero_pred`.
5. Compute `zero_qwk = cohen_kappa_score(labels, zero_preds, weights='quadratic')`.
6. Record `delta_qwk = zero_qwk - full_qwk` (negative = retention branch contributes; near zero = inert).
7. Write per-fold partial CSV under `PARTIAL_DIR`.

ABMIL is included as a control (its zero-retention QWK equals its full QWK by construction).

**Resume-safe**: completed `(encoder, variant, hp, fold)` chunks are skipped on re-run.

In [ ]:
# T1.5 step 6: main loop. Writes per-fold partial CSVs as it goes.
import os, time
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score

# Variants to ablate (the 5 retention-enabled aggregators).
RETENTION_VARIANTS = ['DualBranch', 'RetNet',
                      'SpatialMult', 'SpatialMultRetNet', 'SpatialAdd']
# ABMIL is a single-branch control (no retention; zero-retention QWK == full QWK trivially).
CONTROL_VARIANTS   = ['ABMIL']
ALL_VARIANTS       = CONTROL_VARIANTS + RETENTION_VARIANTS

HPS   = ['default', 'best']
FOLDS = list(range(5))

if TEST_MODE:
    ENCODERS_TO_RUN = ['UNI']
    VARIANTS_TO_RUN = ['RetNet']
    HPS_TO_RUN      = ['best']
    FOLDS_TO_RUN    = [0]
    print('TEST MODE: running RetNet_bestHP / UNI / fold 0 only (~2-3 min).')
else:
    ENCODERS_TO_RUN = ['UNI', 'UNI2-h']
    VARIANTS_TO_RUN = ALL_VARIANTS
    HPS_TO_RUN      = HPS
    FOLDS_TO_RUN    = FOLDS
    n_total = (len(ENCODERS_TO_RUN) * len(VARIANTS_TO_RUN)
               * len(HPS_TO_RUN) * len(FOLDS_TO_RUN))
    print(f'FULL SWEEP: {len(ENCODERS_TO_RUN)}x{len(VARIANTS_TO_RUN)}x'
          f'{len(HPS_TO_RUN)}x{len(FOLDS_TO_RUN)} = '
          f'{n_total} checkpoints x ~1,803 slides each.')


def partial_csv_path(encoder, variant, hp, fold):
    return os.path.join(PARTIAL_DIR,
                        f'{encoder}_{variant}_{hp}HP_fold{fold}.csv')


def process_fold(encoder, variant, hp, fold):
    """Process one (encoder, variant, hp, fold) chunk; write partial CSV; return row dict."""
    out_path = partial_csv_path(encoder, variant, hp, fold)
    if os.path.exists(out_path):
        # Resume: skip already-completed chunks.
        existing = pd.read_csv(out_path)
        if len(existing) > 0:
            return existing.iloc[0].to_dict()

    # 1. Resolve checkpoint.
    ckpt_name = f'{variant}_{hp}HP_fold{fold}_epoch.pt'
    ckpt_path = os.path.join(RESULTS_ROOT, encoder, 'checkpoints', ckpt_name)
    if not os.path.exists(ckpt_path):
        print(f'  SKIP: missing checkpoint {ckpt_path}')
        return None

    # 2. Load cached full-branch predictions + labels.
    slide_ids, full_preds, labels, pred_path = load_predictions_file(
        encoder, variant, hp, fold)
    if slide_ids is None:
        print(f'  SKIP: missing predictions file {pred_path}')
        return None
    full_qwk = cohen_kappa_score(labels, full_preds, weights='quadratic')

    # 3. ABMIL is a control - skip the surgical pass and report full == zero.
    if variant.startswith('ABMIL'):
        row = {
            'encoder': encoder, 'variant': variant, 'hp': hp, 'fold': fold,
            'n_slides': int(len(slide_ids)),
            'n_slides_processed': int(len(slide_ids)),
            'full_qwk': float(full_qwk),
            'zero_retention_qwk': float(full_qwk),     # by construction
            'delta_qwk': 0.0,                          # by construction
            'agreement_full_vs_zero': 1.0,             # by construction
            'note': 'ABMIL control: single-branch, no retention to zero',
        }
        pd.DataFrame([row]).to_csv(out_path, index=False)
        print(f'  CONTROL  {encoder}/{variant}_{hp}HP/fold{fold}: '
              f'full_qwk={full_qwk:.4f} (ABMIL, no retention branch)')
        return row

    # 4. Build model + load checkpoint.
    exp_name  = f'{variant}_{hp}HP'
    input_dim = INPUT_DIMS[encoder]
    model = create_model(exp_name, hp, input_dim).to(device).eval()
    model = load_checkpoint(ckpt_path, model, device)

    # 5. Zero-retention inference.
    features_dir = FEATURE_DIRS[encoder]
    file_ext     = FILE_EXTS[encoder]
    zero_preds   = np.full(len(slide_ids), -1, dtype=int)
    n_processed  = 0
    t0 = time.time()
    for i, sid in enumerate(tqdm(slide_ids,
                                  desc=f'{encoder}/{variant}_{hp}HP/fold{fold}',
                                  leave=False)):
        features, coords = load_slide_features(sid, features_dir, file_ext)
        if features is None:
            continue
        features = features.to(device, non_blocking=True)
        coords   = coords.to(device, non_blocking=True)
        zero_preds[i] = forward_zero_retention(model, features, coords,
                                                temperature=1.0)
        n_processed += 1

    # 6. Free GPU.
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    # 7. Compute QWK only on slides that were processed (zero_pred != -1).
    valid_mask = zero_preds != -1
    if valid_mask.sum() < 2:
        print(f'  WARN: insufficient processed slides for QWK '
              f'({encoder}/{variant}_{hp}HP/fold{fold})')
        return None
    # Recompute full_qwk on the same slide subset to keep the paired comparison fair.
    full_qwk_subset = cohen_kappa_score(
        labels[valid_mask], full_preds[valid_mask], weights='quadratic')
    zero_qwk_subset = cohen_kappa_score(
        labels[valid_mask], zero_preds[valid_mask], weights='quadratic')
    agreement = float(np.mean(full_preds[valid_mask] == zero_preds[valid_mask]))

    row = {
        'encoder': encoder, 'variant': variant, 'hp': hp, 'fold': fold,
        'n_slides': int(len(slide_ids)),
        'n_slides_processed': int(valid_mask.sum()),
        'full_qwk': float(full_qwk_subset),
        'zero_retention_qwk': float(zero_qwk_subset),
        'delta_qwk': float(zero_qwk_subset - full_qwk_subset),
        'agreement_full_vs_zero': agreement,
        'note': '',
    }
    pd.DataFrame([row]).to_csv(out_path, index=False)
    elapsed = time.time() - t0
    print(f'  DONE     {encoder}/{variant}_{hp}HP/fold{fold}: '
          f'full={full_qwk_subset:.4f}, zero={zero_qwk_subset:.4f}, '
          f'delta={zero_qwk_subset - full_qwk_subset:+.4f}, '
          f'agree={agreement*100:.1f}%, n={int(valid_mask.sum())}, '
          f'{elapsed:.1f}s')
    return row


# ----- Main loop -----
all_rows = []
for encoder in ENCODERS_TO_RUN:
    for variant in VARIANTS_TO_RUN:
        for hp in HPS_TO_RUN:
            for fold in FOLDS_TO_RUN:
                row = process_fold(encoder, variant, hp, fold)
                if row is not None:
                    all_rows.append(row)

if all_rows:
    per_fold = pd.DataFrame(all_rows)
    per_fold.to_csv(PER_FOLD_CSV, index=False)
    print(f'\nWrote {len(per_fold)} rows to {PER_FOLD_CSV}')
    per_fold
else:
    print('No chunks processed - check paths above.')

## Cohort-wide aggregation

Pools per-fold rows and computes per `(encoder, variant, hp)`:
- `mean_full_qwk`, `mean_zero_qwk` - 5-fold mean.
- `mean_delta_qwk` = mean of per-fold delta.
- `paired_t_p`, `wilcoxon_p` - paired tests (full vs zero across 5 folds).
- `cohen_d` - paired Cohen's d on the delta.
- `mean_agreement` - fraction of slides where full pred == zero pred.

Interpretation:
- |mean_delta| < 0.001 across all variants  =>  retention branch is empirically inert.
- mean_delta_qwk <= -0.01 (zero-retention drops materially)  =>  retention branch contributes.

In [ ]:
# T1.5 step 7: cohort-wide aggregation.
import os
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

# Defensive: ensure PARTIAL_DIR exists before listdir (in case Cell 2 didn't run
# or the Drive mount happened in a different session).
os.makedirs(PARTIAL_DIR, exist_ok=True)

if not os.path.exists(PER_FOLD_CSV):
    # Rebuild from partial files if main loop wrote partials but not the combined csv.
    files = [f for f in sorted(os.listdir(PARTIAL_DIR)) if f.endswith('.csv')]
    if not files:
        raise FileNotFoundError(
            f'No partial CSVs found in {PARTIAL_DIR}. '
            f'Did Cell 10 (main loop) run successfully? If TEST_MODE=True, '
            f'check that ABMIL_bestHP UNI fold 0 inference produced a partial CSV. '
            f'You can also re-run Cell 10 after fixing the upstream error.')
    parts = [pd.read_csv(os.path.join(PARTIAL_DIR, f)) for f in files]
    per_fold = pd.concat(parts, ignore_index=True)
    per_fold.to_csv(PER_FOLD_CSV, index=False)
else:
    per_fold = pd.read_csv(PER_FOLD_CSV)
print(f'Loaded {len(per_fold)} per-fold rows from {PER_FOLD_CSV}.')


def summarise(group):
    full   = group['full_qwk'].to_numpy(dtype=float)
    zero   = group['zero_retention_qwk'].to_numpy(dtype=float)
    delta  = zero - full
    n      = len(delta)
    # Paired t-test (only meaningful with >=2 folds).
    if n >= 2:
        t_stat, t_p = scipy_stats.ttest_rel(full, zero)
        try:
            w_stat, w_p = scipy_stats.wilcoxon(full, zero)
        except ValueError:
            w_stat, w_p = float('nan'), float('nan')
        # Cohen's d (paired): mean(delta) / std(delta).
        d_std = float(np.std(delta, ddof=1))
        cohen_d = float(np.mean(delta) / d_std) if d_std > 0 else float('nan')
    else:
        t_stat, t_p, w_stat, w_p, cohen_d = (float('nan'),) * 5
    return pd.Series({
        'n_folds':         int(n),
        'mean_full_qwk':   float(np.mean(full)),
        'std_full_qwk':    float(np.std(full,  ddof=1)) if n > 1 else float('nan'),
        'mean_zero_qwk':   float(np.mean(zero)),
        'std_zero_qwk':    float(np.std(zero,  ddof=1)) if n > 1 else float('nan'),
        'mean_delta_qwk':  float(np.mean(delta)),
        'std_delta_qwk':   float(np.std(delta, ddof=1)) if n > 1 else float('nan'),
        'min_delta_qwk':   float(np.min(delta)),
        'max_delta_qwk':   float(np.max(delta)),
        'paired_t_stat':   float(t_stat) if not np.isnan(t_stat) else float('nan'),
        'paired_t_p':      float(t_p)    if not np.isnan(t_p)    else float('nan'),
        'wilcoxon_p':      float(w_p)    if not np.isnan(w_p)    else float('nan'),
        'cohen_d_paired':  cohen_d,
        'mean_agreement':  float(group['agreement_full_vs_zero'].mean()),
    })


summary = (per_fold.groupby(['encoder', 'variant', 'hp'], dropna=False)
                   .apply(summarise)
                   .reset_index())
summary.to_csv(SUMMARY_CSV, index=False)
print(f'Wrote summary to {SUMMARY_CSV}')
summary

## Output for thesis

Prints the defence-ready sentence per `(encoder, variant, hp)` and the headline interpretation. Paste into Ch.4 or Ch.5 (Discussion) once the full sweep completes.

In [ ]:
# T1.5 step 8: thesis-ready output strings.
import pandas as pd
import numpy as np

if 'summary' not in dir():
    summary = pd.read_csv(SUMMARY_CSV)

INERT_DELTA_THRESHOLD     = 0.001   # |delta| < this  =>  branch effectively inert
MATERIAL_DELTA_THRESHOLD  = -0.01   # delta <= this    =>  branch contributes materially


def row(enc, var, hp):
    m = ((summary['encoder'] == enc)
         & (summary['variant'] == var)
         & (summary['hp']      == hp))
    return summary[m].iloc[0] if m.any() else None


def anchor(enc, var, hp):
    r = row(enc, var, hp)
    if r is None:
        return f'  {enc}/{var}_{hp}HP: [not yet processed]'
    return (
        f"  {enc}/{var}_{hp}HP (n_folds={int(r['n_folds'])}): "
        f"full QWK = {r['mean_full_qwk']:.4f} +/- {r['std_full_qwk']:.4f}, "
        f"zero-retention QWK = {r['mean_zero_qwk']:.4f} +/- {r['std_zero_qwk']:.4f}, "
        f"Delta QWK = {r['mean_delta_qwk']:+.4f} (paired-t p = {r['paired_t_p']:.4f}, "
        f"Wilcoxon p = {r['wilcoxon_p']:.4f}, Cohen's d = {r['cohen_d_paired']:+.3f}), "
        f"slide-level agreement = {r['mean_agreement']*100:.1f}%."
    )


print('=' * 90)
print('T1.5 PER-CELL ABLATION ANCHORS - paste into Ch.4 (Results) / Ch.5 (Discussion)')
print('=' * 90)
for enc in ['UNI', 'UNI2-h']:
    for hp in ['best', 'default']:
        print(f'\n[{enc} / {hp}HP]')
        for var in ['ABMIL', 'DualBranch', 'RetNet',
                    'SpatialMult', 'SpatialMultRetNet', 'SpatialAdd']:
            print(anchor(enc, var, hp))

# --- Headline configuration: RetNet_bestHP on UNI (the thesis-cited best config). ---
print('\n' + '=' * 90)
print('HEADLINE - RetNet bestHP / UNI')
print('=' * 90)
h = row('UNI', 'RetNet', 'best')
if h is not None:
    print(
        f"RetNet_bestHP on UNI: full-branch 5-fold mean QWK = {h['mean_full_qwk']:.4f}, "
        f"zero-retention QWK = {h['mean_zero_qwk']:.4f}, "
        f"Delta QWK = {h['mean_delta_qwk']:+.4f} "
        f"(paired-t p = {h['paired_t_p']:.4f}, Cohen's d = {h['cohen_d_paired']:+.3f})."
    )
else:
    print('  [RetNet/best/UNI not yet processed]')

# --- Verdict across all retention-enabled cells. ---
print('\n' + '=' * 90)
print('VERDICT')
print('=' * 90)
retention_rows = summary[summary['variant'].isin(
    ['DualBranch', 'RetNet', 'SpatialMult', 'SpatialMultRetNet', 'SpatialAdd'])]
if len(retention_rows) == 0:
    print('  No retention-enabled cells processed yet.')
else:
    deltas = retention_rows['mean_delta_qwk'].to_numpy(dtype=float)
    n_inert    = int(np.sum(np.abs(deltas) < INERT_DELTA_THRESHOLD))
    n_material = int(np.sum(deltas <= MATERIAL_DELTA_THRESHOLD))
    n_cells    = int(len(deltas))
    print(f'  Total retention-enabled cells processed: {n_cells}')
    print(f'  Cells where |Delta QWK| < {INERT_DELTA_THRESHOLD} '
          f'(effectively inert): {n_inert} / {n_cells}')
    print(f'  Cells where Delta QWK <= {MATERIAL_DELTA_THRESHOLD} '
          f'(branch contributes materially): {n_material} / {n_cells}')
    print(f'  Delta range across cells: [{deltas.min():+.4f}, {deltas.max():+.4f}], '
          f'mean = {deltas.mean():+.4f}')
    if n_inert == n_cells:
        print('\n  CONCLUSION: zero-retention QWK is within +/-0.001 of full-branch QWK in')
        print('  every cell. The retention branch contribution is EMPIRICALLY NEGLIGIBLE')
        print('  in the trained model state, consistent with the rank-1 attention dominance')
        print('  reported in Section 3.6. The thesis architectural claim should be qualified.')
    elif n_material >= n_cells // 2:
        print('\n  CONCLUSION: zero-retention QWK drops materially (Delta <= -0.01) in')
        print('  the majority of cells. The retention branch CONTRIBUTES, supporting the')
        print('  thesis architectural claim.')
    else:
        print('\n  CONCLUSION: mixed - the retention branch contributes in some cells but')
        print('  not others. See per-cell anchors above; the thesis should report the')
        print('  per-variant / per-encoder pattern rather than a single headline.')